# Model Comparison Notebook

Compares **Unmatched** distance (eigenvalues of the normalised Gram matrix) and **Wasserstein-2** distance (sorted scalar products of EDRep embeddings) on graphs generated by three synthetic models:

- **SBM** — Degree-Corrected Stochastic Block Model: `α` ∈ {0.85, …, 1.75}, k=2 communities, n=1,000 nodes
- **CM** — Configuration Model via DCSBM: `α` ∈ {0.70, …, 3.00}, k=1 community, degree weights θ = Uniform(3,10)^α
- **GM** — Geometric Model on the unit disk: `β` ∈ {1.20, …, 3.08}, connection probability ∝ exp(−β·d)

**Pipeline per model:** generate graphs → compute EDRep embeddings → build pairwise distance matrices → scatter plot (Unmatched vs Wasserstein) → NMI curves vs parameter.

The notebook is organised into three pipeline stages of increasing scale:

| Section | Description | n |
|---|---|---|
| 1–3 | Single-run  | 1,000 | 
| 4 | Multiple Realization  | 1,000 | 
| 5 | Large-graph MC  | 10,000 |

In [27]:
import sys
import numpy as np
import random
import itertools
import pandas as pd
import os
import ot
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.sparse import csr_matrix
from scipy.spatial.distance import squareform
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF
from sklearn.metrics.cluster import normalized_mutual_info_score as NMI
from sklearn.metrics.pairwise import euclidean_distances
from EDRep_main.EDRep import NodeEmbedding
from tqdm.notebook import tqdm
from matplotlib.ticker import FormatStrFormatter
from graph_generators import *

---
## Helper functions

Core graph generators and utilities used throughout all stages.

| Function | Description |
|---|---|
| `generateSequence` | Generates `n_graphs` graphs with a given model and saves each as a CSV edge list |
| `DCSBM` | Degree-Corrected SBM — samples an edge list given block matrix `C`, labels `ℓ`, and degree weights `θ` |
| `GeometricModel` | Random geometric graph on the unit disk — connection probability ∝ `exp(−β·d)` |
| `gen_cm_via_dcsbm` | Configuration Model via DCSBM with a single community and power-law degree weights |
| `compute_cin_cout` | Maps `(c, k, α)` to within/between-community edge rates `c_in`, `c_out` |
| `EmbDistance` | Unmatched (default) or matched distance between two embedding matrices |
| `get_adj` / `df_to_adj` | Load a CSV or DataFrame edge list and return a sparse CSR adjacency matrix |
| `node_embedding` | Thin wrapper around `NodeEmbedding` with fixed seed and NumPy state restoration |
| `w2_distance` | Wasserstein-2 between two embeddings via their flattened Gram matrices (single pair) |
| `create_embeddings` | Batch embedding over a list of CSV files |
| `NMF_kmeans` / `ClusterNMF` | NMF-based spectral clustering with 20 random restarts (minimises KMeans inertia) |
| `precompute_features` | Caches sorted upper-triangular scalar products and eigenvalue arrays for fast distance computation |
| `compute_distance_matrices` | Builds pairwise Unmatched and Wasserstein distance matrices; returns a tidy DataFrame of pair-type labels |

In [ ]:
def generateSequence(outputfolder, model, args, n_graphs, start_idx=1, verbose=True, append_name='.csv'):
    for i in range(n_graphs):
        df = model(args)
        idx_str = str(start_idx + i).zfill(2)
        df.to_csv(outputfolder + '/EL' + idx_str + append_name, index=False)
        if verbose:
            print("[%--25s] %d%%" % ('='*(int(i/(n_graphs)*25)) + '>', (i+1)/(n_graphs)*100), end='\r')
    return


def EmbDistance(X, Y, distance_type='unmatched'):
    n1, d1 = X.shape
    n2, d2 = Y.shape
    if d1 != d2:
        raise ValueError('Embedding dimensions differ')
    if distance_type == 'matched':
        if n1 != n2:
            raise ValueError('Matrix sizes differ for matched distance')
        Mxx = X.T@X; Mxy = X.T@Y; Myy = Y.T@Y
        return np.sqrt(np.abs(np.linalg.norm(Mxx)**2 + np.linalg.norm(Myy)**2 - 2*np.linalg.norm(Mxy)**2))
    else:
        lam1 = np.linalg.eigvalsh(X.T@X) / n1
        lam2 = np.linalg.eigvalsh(Y.T@Y) / n2
        return np.linalg.norm(lam1 - lam2)


def get_adj(filepath):
    df = pd.read_csv(filepath)
    G = nx.from_pandas_edgelist(df, 'i', 'j')
    return nx.adjacency_matrix(G).astype(float)


def node_embedding(g, dim_embedding, seed=42):
    rdn_state = np.random.get_state()
    random.seed(seed); np.random.seed(seed)
    X_total = NodeEmbedding(g, dim=dim_embedding, k=1, verbose=False).X
    np.random.set_state(rdn_state)
    return X_total


def w2_distance(emb_1, emb_2):
    S_1 = emb_1@emb_1.T; S_2 = emb_2@emb_2.T
    return ot.wasserstein_1d(S_1.flatten(), S_2.flatten(), p=2)**(1/2)


def create_embeddings(path_files, dim):
    embeddings = []
    for file in tqdm(path_files, desc="Computing embeddings"):
        embeddings.append(node_embedding(get_adj(file), dim))
    return embeddings


def precompute_features(embeddings):
    eigenvalues, S_sorted_list = [], []
    for X in tqdm(embeddings, desc="Pre-computing features", leave=False):
        n = len(X)
        eigenvalues.append(np.linalg.eigvalsh(X.T @ X) / n)
        X32 = X.astype(np.float32)
        iu = np.triu_indices(n)
        S_sorted_list.append(np.sort((X32 @ X32.T)[iu]))
    return eigenvalues, S_sorted_list


def compute_distance_matrices(all_emb, labels, model_name):
    N = len(all_emb)
    pairs = list(itertools.combinations(range(N), 2))

    eigenvalues, S_sorted_list = precompute_features(all_emb)

    eig_matrix = np.stack(eigenvalues)
    D_unmatched = euclidean_distances(eig_matrix)

    w2_vals = []
    for i, j in tqdm(pairs, desc=f"{model_name} W2", leave=True):
        s1, s2 = S_sorted_list[i], S_sorted_list[j]
        if len(s1) == len(s2):
            w2_vals.append(np.sqrt(np.mean((s1 - s2) ** 2)))
        else:
            w2_vals.append(ot.wasserstein_1d(s1, s2, p=2) ** 0.5)

    D_wasserstein = squareform(w2_vals)

    categories = []
    for i, j in pairs:
        a_i, a_j = labels[i], labels[j]
        if a_i == a_j:
            categories.append(f"{model_name} {a_i:.2f}")
        else:
            a_min, a_max = sorted([a_i, a_j])
            categories.append(f"{a_min:.2f} vs {a_max:.2f}")

    df = pd.DataFrame({
        'Unmatched':   [D_unmatched[i, j] for i, j in pairs],
        'Wasserstein': w2_vals,
        'Type':        categories,
    })
    return df, D_unmatched, D_wasserstein

---
## Configuration

Shared parameters used across all three pipeline stages.

| Parameter | Value | Description |
|---|---|---|
| `n_nodes` | 1,000 | Number of nodes per graph |
| `c` | 10 | Target average degree |
| `dim` | 32 | EDRep embedding dimension |
| `n_graphs` | 50 | Graphs per parameter value |
| `symmetric` | True | Symmetrise the generated edge list |
| `make_connected` | True | Force graph connectivity by attaching isolated nodes to the highest-degree node |

In [ ]:
# Shared parameters across all models
n_nodes = 1000
gamma = 0.8
symmetric = True
make_connected = True
c = 10
dim = 32
n_graphs = 50  # graphs per parameter value

sbm_alphas = [0.85, 0.94, 1.04, 1.16, 1.28, 1.42, 1.58, 1.75]
cm_alphas  = [0.70, 1.06, 1.31, 1.45, 1.61, 1.98, 2.30 ,2.44, 3.00]
gm_betas   = [1.20, 1.40, 1.64, 1.92, 2.10,2.25, 2.42, 2.63, 3.08]

### Output directories

Creates the folder tree for CSV edge lists under `data/synthetic_graphs/`: one sub-folder per parameter value for each model.

In [ ]:
base_sbm = os.path.join("data", "synthetic_graphs", "generated_sbm")
base_cm  = os.path.join("data", "synthetic_graphs", "generated_cm")
base_gm  = os.path.join("data", "synthetic_graphs", "generated_gm")

for a in sbm_alphas2:
    os.makedirs(os.path.join(base_sbm, f"alpha_{a:.2f}"), exist_ok=True)

for a in cm_alphas:
    os.makedirs(os.path.join(base_cm, f"alpha_{a:.2f}"), exist_ok=True)

for b in gm_betas:
    os.makedirs(os.path.join(base_gm, f"beta_{b}"), exist_ok=True)

print("Folders created.")

---
## 1. Stochastic Block Model (SBM)

**Parameter**: `α` ∈ {0.85, 0.94, 1.04, 1.16, 1.28, 1.42, 1.58, 1.75} — k=2 communities, n=1,000 nodes.

`α` controls the signal-to-noise ratio of the block structure via:
- `c_out = c − α·√c` (between-community edge rate)
- `c_in  = k·c − (k−1)·c_out` (within-community edge rate)

Low `α` → weak separation (close to Erdős–Rényi).\
High `α` → sharp community boundaries.

**Steps:** generate 50 graphs per `α` → embed → pairwise distances → scatter plot → NMI curve.

In [ ]:
k_sbm = 2
theta_sbm = np.ones(n_nodes)
l_sbm = np.zeros(n_nodes)
for i in range(k_sbm):
    l_sbm[i*(n_nodes//k_sbm):(i+1)*(n_nodes//k_sbm)] = i
l_sbm = l_sbm.astype(int)

for a in sbm_alphas2:
    c_in, c_out = compute_cin_cout(c, k_sbm, a)
    C_sbm = np.ones((k_sbm, k_sbm)) * c_out + np.diag(np.ones(k_sbm)) * (c_in - c_out)
    folder = os.path.join(base_sbm, f"alpha_{a:.2f}")
    print(f"Generating SBM alpha={a:.2f} -> {folder}")
    generateSequence(folder, DCSBM, (C_sbm, c, l_sbm, theta_sbm, symmetric, make_connected),
                     n_graphs, start_idx=1, verbose=True)
    print()

In [ ]:
embeddings_sbm = {}
for a in sbm_alphas2:
    folder = os.path.join(base_sbm, f"alpha_{a:.2f}")
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    print(f"Embedding SBM alpha={a:.2f} ({len(files)} graphs)...")
    embeddings_sbm[a] = create_embeddings(files, dim)

In [ ]:
all_emb_sbm, labels_sbm = [], []
for a, emb_list in embeddings_sbm.items():
    for emb in emb_list:
        all_emb_sbm.append(emb); labels_sbm.append(a)

df_sbm, D_unmatched_sbm, D_wasserstein_sbm = compute_distance_matrices(
    all_emb_sbm, labels_sbm, "SBM")

In [ ]:
sbm_keep = ["SBM 0.00", "0.00 vs 0.50", "0.00 vs 0.75", "0.00 vs 1.00",
            "0.00 vs 1.25", "0.00 vs 1.50", "0.00 vs 1.75", "0.00 vs 2.00"]
df_sbm_f = df_sbm[df_sbm['Type'].isin(sbm_keep)]

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=df_sbm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=sbm_keep, palette='tab10', alpha=0.7, s=50, ax=ax)
ax.set_title("SBM: Unmatched vs Wasserstein", fontsize=15)
ax.set_xlabel("Unmatched", fontsize=12); ax.set_ylabel("Wasserstein", fontsize=12)
ax.legend(title="alpha pair", bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
l_true_sbm = np.concatenate([[i]*n_graphs for i in range(len(sbm_alphas2))])
k_all_sbm = len(sbm_alphas2)
nmi_all_u_sbm = NMI(ClusterNMF(D_unmatched_sbm, k_all_sbm), l_true_sbm)
nmi_all_w_sbm = NMI(ClusterNMF(D_wasserstein_sbm, k_all_sbm), l_true_sbm)
print(f"SBM global NMI — Unmatched:   {nmi_all_u_sbm:.4f}")
print(f"SBM global NMI — Wasserstein: {nmi_all_w_sbm:.4f}")

In [ ]:
nmi_u_sbm_curve, nmi_w_sbm_curve = [], []

for i, a in enumerate(sbm_alphas2):
    if i == 0:
        nmi_u_sbm_curve.append(0.0); nmi_w_sbm_curve.append(0.0); continue
    idx = np.concatenate([np.arange(0, n_graphs), np.arange(i*n_graphs, (i+1)*n_graphs)])
    l_sub = np.concatenate([np.zeros(n_graphs), np.ones(n_graphs)])
    nmi_u_sbm_curve.append(NMI(ClusterNMF(D_unmatched_sbm[np.ix_(idx,idx)], k=2), l_sub))
    nmi_w_sbm_curve.append(NMI(ClusterNMF(D_wasserstein_sbm[np.ix_(idx,idx)], k=2), l_sub))

plt.figure(figsize=(8, 5))
plt.plot(sbm_alphas2, nmi_u_sbm_curve, label='Unmatched', marker='o', color='steelblue')
plt.plot(sbm_alphas2, nmi_w_sbm_curve, label='Wasserstein', marker='s', color='darkorange')
plt.title("SBM: NMI vs alpha")
plt.xlabel('alpha')
plt.ylabel('NMI')
plt.ylim(-0.05, 1.05)
plt.xticks(sbm_alphas2)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

---
## 2. Configuration Model (CM)

**Parameter**: `α` ∈ {0.70, 1.06, 1.31, 1.45, 1.61, 1.98, 2.30, 2.44, 3.00} — k=1 community, n=1,000 nodes.

Degree weights are drawn as `θ ~ Uniform(3, 10)^α` and then normalised to mean 1. `DCSBM` is called with a single-block connectivity matrix `C = [[c]]` and all nodes in community 0.

- Low `α` → nearly homogeneous degrees (close to Erdős–Rényi).
- High `α` → heavier-tailed degree distribution with more pronounced hubs.

**Steps:** generate 50 graphs per `α` → embed → pairwise distances → scatter plot → NMI curve.

In [ ]:
for a in cm_alphas:
    folder = os.path.join(base_cm, f"alpha_{a:.2f}")
    print(f"Generating CM alpha={a:.2f} -> {folder}")
    generateSequence(folder, gen_cm_via_dcsbm,
                     (n_nodes, c, a, symmetric, make_connected),
                     n_graphs, start_idx=1, verbose=True)
    print()

In [ ]:
embeddings_cm = {}
for a in cm_alphas:
    folder = os.path.join(base_cm, f"alpha_{a:.2f}")
    files = sorted(glob.glob(os.path.join(folder, '*.csv')))
    print(f"Embedding CM alpha={a:.2f} ({len(files)} graphs)...")
    embeddings_cm[a] = create_embeddings(files, dim)

In [ ]:
all_emb_cm, labels_cm = [], []
for a, emb_list in embeddings_cm.items():
    for emb in emb_list:
        all_emb_cm.append(emb); labels_cm.append(a)

df_cm, D_unmatched_cm, D_wasserstein_cm = compute_distance_matrices(
    all_emb_cm, labels_cm, "CM")

In [ ]:
cm_keep = ["CM 0.00", "0.00 vs 0.50", "0.00 vs 1.00", "0.00 vs 1.50",
           "0.00 vs 2.00", "0.00 vs 2.50", "0.00 vs 3.00"]
df_cm_f = df_cm[df_cm['Type'].isin(cm_keep)]

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=df_cm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=cm_keep, palette='tab10', alpha=0.7, s=50, ax=ax)
ax.set_title("CM: Unmatched vs Wasserstein", fontsize=15)
ax.set_xlabel("Unmatched", fontsize=12); ax.set_ylabel("Wasserstein", fontsize=12)
ax.legend(title='alpha pair', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
l_true_cm = np.concatenate([[i]*n_graphs for i in range(len(cm_alphas))])
k_all_cm = len(cm_alphas)
nmi_all_u_cm = NMI(ClusterNMF(D_unmatched_cm, k_all_cm), l_true_cm)
nmi_all_w_cm = NMI(ClusterNMF(D_wasserstein_cm, k_all_cm), l_true_cm)
print(f"CM global NMI — Unmatched:   {nmi_all_u_cm:.4f}")
print(f"CM global NMI — Wasserstein: {nmi_all_w_cm:.4f}")

In [ ]:
nmi_u_cm_curve, nmi_w_cm_curve = [], []

for i, a in enumerate(cm_alphas):
    if i == 0:
        nmi_u_cm_curve.append(0.0); nmi_w_cm_curve.append(0.0); continue
    idx = np.concatenate([np.arange(0, n_graphs), np.arange(i*n_graphs, (i+1)*n_graphs)])
    l_sub = np.concatenate([np.zeros(n_graphs), np.ones(n_graphs)])
    nmi_u_cm_curve.append(NMI(ClusterNMF(D_unmatched_cm[np.ix_(idx,idx)], k=2), l_sub))
    nmi_w_cm_curve.append(NMI(ClusterNMF(D_wasserstein_cm[np.ix_(idx,idx)], k=2), l_sub))

plt.figure(figsize=(8, 5))
plt.plot(cm_alphas, nmi_u_cm_curve, label='Unmatched', marker='o', color='steelblue')
plt.plot(cm_alphas, nmi_w_cm_curve, label='Wasserstein', marker='s', color='darkorange')
plt.title("CM: NMI vs alpha")
plt.xlabel('alpha')
plt.ylabel('NMI')
plt.ylim(-0.05, 1.05)
plt.xticks(cm_alphas)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

---
## 3. Geometric Model (GM)

**Parameter**: `β` ∈ {1.20, 1.40, 1.64, 1.92, 2.10, 2.25, 2.42, 2.63, 3.08} — n=1,000 nodes placed uniformly on the unit disk.

Connection probability between nodes `i` and `j` is proportional to `exp(−β·d(i,j))`, where `d` is Euclidean distance. Each node samples `c/2` neighbours from this distribution.

- Low `β` → spatially uniform connections (close to Erdős–Rényi).
- High `β` → strongly local connections, leading to geographic clusters.

**Steps:** generate 50 graphs per `β` → embed → pairwise distances → scatter plot → NMI curve.

In [ ]:
for b in gm_betas:
    folder = os.path.join(base_gm, f"beta_{b}")
    n_gm = n_nodes  # fixed size for all beta values
    r_gm = np.random.uniform(0, 1, n_gm)
    theta_gm = np.random.uniform(0, 2*np.pi, n_gm)
    X_gm = np.column_stack([r_gm*np.cos(theta_gm), r_gm*np.sin(theta_gm)])
    print(f"Generating GM beta={b} -> {folder}")
    generateSequence(folder, GeometricModel, (X_gm, c, b),
                     n_graphs, start_idx=1, verbose=True)
    print()

In [ ]:
embeddings_gm = {}
for b in gm_betas:
    folder = os.path.join(base_gm, f"beta_{b}")
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    print(f"Embedding GM beta={b} ({len(files)} graphs)...")
    embeddings_gm[b] = create_embeddings(files, dim)

In [ ]:
all_emb_gm, labels_gm = [], []
for b, emb_list in embeddings_gm.items():
    for emb in emb_list:
        all_emb_gm.append(emb); labels_gm.append(b)

df_gm, D_unmatched_gm, D_wasserstein_gm = compute_distance_matrices(
    all_emb_gm, labels_gm, "GM")

In [ ]:
gm_keep = ["GM 0.00", "0.00 vs 0.50", "0.00 vs 1.00", "0.00 vs 1.50", "0.00 vs 2.00",
           "0.00 vs 2.50", "0.00 vs 3.00", "0.00 vs 3.50", "0.00 vs 4.00"]
df_gm_f = df_gm[df_gm['Type'].isin(gm_keep)]

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=df_gm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=gm_keep, palette='tab10', alpha=0.7, s=50, ax=ax)
ax.set_title("GM: Unmatched vs Wasserstein", fontsize=15)
ax.set_xlabel("Unmatched", fontsize=12); ax.set_ylabel("Wasserstein", fontsize=12)
ax.legend(title='beta pair', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
l_true_gm = np.concatenate([[i]*n_graphs for i in range(len(gm_betas))])
k_all_gm = len(gm_betas)
nmi_all_u_gm = NMI(ClusterNMF(D_unmatched_gm, k_all_gm), l_true_gm)
nmi_all_w_gm = NMI(ClusterNMF(D_wasserstein_gm, k_all_gm), l_true_gm)
print(f"GM global NMI — Unmatched:   {nmi_all_u_gm:.4f}")
print(f"GM global NMI — Wasserstein: {nmi_all_w_gm:.4f}")

In [ ]:
nmi_u_gm_curve, nmi_w_gm_curve = [], []

for i, b in enumerate(gm_betas):
    if i == 0:
        nmi_u_gm_curve.append(0.0); nmi_w_gm_curve.append(0.0); continue
    idx = np.concatenate([np.arange(0, n_graphs), np.arange(i*n_graphs, (i+1)*n_graphs)])
    l_sub = np.concatenate([np.zeros(n_graphs), np.ones(n_graphs)])
    nmi_u_gm_curve.append(NMI(ClusterNMF(D_unmatched_gm[np.ix_(idx,idx)], k=2), l_sub))
    nmi_w_gm_curve.append(NMI(ClusterNMF(D_wasserstein_gm[np.ix_(idx,idx)], k=2), l_sub))

plt.figure(figsize=(8, 5))
plt.plot(gm_betas, nmi_u_gm_curve, label='Unmatched', marker='o', color='steelblue')
plt.plot(gm_betas, nmi_w_gm_curve, label='Wasserstein', marker='s', color='darkorange')
plt.title("GM: NMI vs beta")
plt.xlabel('beta')
plt.ylabel('NMI')
plt.ylim(-0.05, 1.05)
plt.xticks(gm_betas)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

---
## Summary — Combined comparison across all models

Side-by-side scatter plots (Unmatched vs Wasserstein) and NMI curves for SBM, CM, and GM, placed together to compare the discriminative power of the two distance metrics across graph families.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(30, 7))
fig.suptitle("Scatter Unmatched vs Wasserstein — SBM, CM, GM", fontsize=15)

sns.scatterplot(data=df_sbm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=sbm_keep, palette='tab10', alpha=0.7, s=40, ax=axes[0])
axes[0].set_title("SBM", fontsize=13)
axes[0].set_xlabel('Unmatched'); axes[0].set_ylabel('Wasserstein')
axes[0].legend(title='alpha pair', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.scatterplot(data=df_cm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=cm_keep, palette='tab10', alpha=0.7, s=40, ax=axes[1])
axes[1].set_title("CM", fontsize=13)
axes[1].set_xlabel('Unmatched'); axes[1].set_ylabel('Wasserstein')
axes[1].legend(title='alpha pair', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[1].grid(True, linestyle='--', alpha=0.5)

sns.scatterplot(data=df_gm_f, x='Unmatched', y='Wasserstein',
                hue='Type', hue_order=gm_keep, palette='tab10', alpha=0.7, s=40, ax=axes[2])
axes[2].set_title("GM", fontsize=13)
axes[2].set_xlabel('Unmatched'); axes[2].set_ylabel('Wasserstein')
axes[2].legend(title='beta pair', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
fig.suptitle("NMI vs Parametro — SBM, CM, GM", fontsize=15)

model_configs = [
    ("SBM",  sbm_alphas2,  nmi_u_sbm_curve,  nmi_w_sbm_curve,  "alpha"),
    ("CM",   cm_alphas,   nmi_u_cm_curve,   nmi_w_cm_curve,   "alpha"),
    ("GM",   gm_betas,    nmi_u_gm_curve,   nmi_w_gm_curve,   "beta"),
]

for ax, (name, params, nmi_u, nmi_w, xlabel) in zip(axes.ravel(), model_configs):
    ax.plot(params, nmi_u, marker="o", color="steelblue",  linestyle="-",  label="Unmatched")
    ax.plot(params, nmi_w, marker="s", color="darkorange", linestyle="--", label="Wasserstein")
    ax.set_title(f"{name}: NMI vs {xlabel}", fontsize=13)
    ax.set_xlabel(xlabel); ax.set_ylabel('NMI')
    ax.set_xticks(params); ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(fontsize=10)

plt.tight_layout(); plt.show()

---
## Section 4 — Realization-based pipeline (n=1,000)

Repeats the full experiment over **10 independent realizations** to assess variability. Each realization is saved to `output_realizations_emb/real_XX.npz` with one key per `(model, param)` pair (e.g. `sbm_a0.85`, `cm_a1.06`, `gm_b1.40`).

**Checkpointing:** `load_or_compute_emb` reads the existing NPZ (if any) and computes only the keys that are absent — adding a new `α`/`β` value re-runs only the missing graphs, leaving the rest intact.

In [ ]:
def _embed_for_sbm(a, seed):
    k2 = 2
    l2 = np.array([i // (n_nodes // k2) for i in range(n_nodes)], dtype=int)
    theta2 = np.ones(n_nodes)
    c_in, c_out = compute_cin_cout(c, k2, a)
    C2 = np.ones((k2, k2)) * c_out + np.diag(np.ones(k2)) * (c_in - c_out)
    np.random.seed(seed)
    embs = []
    for _ in range(n_graphs):
        df = DCSBM((C2, c, l2, theta2, symmetric, make_connected))
        embs.append(node_embedding(df_to_adj(df), dim))
        del df
    return np.stack(embs).astype(np.float32)


def _embed_for_cm(a, seed):
    np.random.seed(seed)
    embs = []
    for _ in range(n_graphs):
        df = gen_cm_via_dcsbm((n_nodes, c, a, symmetric, make_connected))
        embs.append(node_embedding(df_to_adj(df), dim))
        del df
    return np.stack(embs).astype(np.float32)


def _embed_for_gm(b, seed):
    np.random.seed(seed)
    r_xy = np.random.uniform(0, 1, n_nodes)
    th   = np.random.uniform(0, 2 * np.pi, n_nodes)
    X_pos = np.column_stack([r_xy * np.cos(th), r_xy * np.sin(th)])
    embs = []
    for _ in range(n_graphs):
        df = GeometricModel((X_pos, c, b))
        embs.append(node_embedding(df_to_adj(df), dim))
        del df
    return np.stack(embs).astype(np.float32)


def load_or_compute_emb(r, out_dir_emb):
    """Load saved embeddings for realization r; compute and cache only missing (model, param) keys."""
    npz_path = os.path.join(out_dir_emb, f'real_{r:02d}.npz')
    saved = {}
    if os.path.exists(npz_path):
        saved = dict(np.load(npz_path, allow_pickle=True))

    updated = False

    for a in sbm_alphas:
        key = f'sbm_a{float(a):.2f}'
        if key not in saved:
            seed = r * 1_000 + int(round(a * 1000))
            print(f'  real_{r:02d}: SBM α={float(a):.2f}  (seed={seed})...')
            saved[key] = _embed_for_sbm(a, seed)
            updated = True

    for a in cm_alphas:
        key = f'cm_a{float(a):.2f}'
        if key not in saved:
            seed = r * 1_000+ 100 + int(round(a * 1000))
            print(f'  real_{r:02d}: CM α={float(a):.2f}  (seed={seed})...')
            saved[key] = _embed_for_cm(a, seed)
            updated = True

    for b in gm_betas:
        key = f'gm_b{float(b):.2f}'
        if key not in saved:
            seed = r * 1_000 + 200 + int(round(b * 1000))
            print(f'  real_{r:02d}: GM β={float(b):.2f}  (seed={seed})...')
            saved[key] = _embed_for_gm(b, seed)
            updated = True

    if updated:
        np.savez_compressed(npz_path, **saved)
        print(f'  → saved {npz_path}')

    return saved

In [ ]:
n_real_emb  = 10
out_dir_emb = 'output_realizations_emb'
os.makedirs(out_dir_emb, exist_ok=True)

for r in tqdm(range(n_real_emb), desc='Realizations'):
    load_or_compute_emb(r, out_dir_emb)

print('Done.')


### Distance matrices from cached embeddings

Loads each realization NPZ, concatenates embeddings across all parameter values for a given model, and computes pairwise Unmatched and Wasserstein distance matrices via `distances_from_emb_stack`. Results across the 10 realizations are averaged and stored in `means_emb`.

In [7]:
def distances_from_emb_stack(X_stack):
    """Unmatched and Wasserstein distance matrices from (N, n, d) embedding stack."""
    N, n, d = X_stack.shape
    pairs = list(itertools.combinations(range(N), 2))

    eigvals = [np.linalg.eigvalsh(X.T @ X) / n for X in X_stack]

    S_sorted = [
        np.sort((X.astype(np.float32) @ X.astype(np.float32).T)[np.triu_indices(n, k=1)])
        for X in X_stack
    ]

    unm_vals, w2_vals = [], []
    for i, j in pairs:
        unm_vals.append(float(np.linalg.norm(eigvals[i] - eigvals[j])))
        s1, s2 = S_sorted[i], S_sorted[j]
        w2_vals.append(float(np.sqrt(np.mean((s1 - s2) ** 2))))

    return squareform(unm_vals), squareform(w2_vals)


In [ ]:
files_emb = sorted(glob.glob(os.path.join(out_dir_emb, 'real_*.npz')))
n_real_loaded = len(files_emb)

D_u_sbm_all, D_w_sbm_all = [], []
D_u_cm_all,  D_w_cm_all  = [], []
D_u_gm_all,  D_w_gm_all  = [], []

for f in tqdm(files_emb, desc='Computing distances'):
    saved = dict(np.load(f, allow_pickle=True))

    X_sbm = np.concatenate([saved[f'sbm_a{float(a):.2f}'] for a in sbm_alphas2], axis=0)
    Du, Dw = distances_from_emb_stack(X_sbm)
    D_u_sbm_all.append(Du); D_w_sbm_all.append(Dw)

    X_cm = np.concatenate([saved[f'cm_a{float(a):.2f}'] for a in cm_alphas], axis=0)
    Du, Dw = distances_from_emb_stack(X_cm)
    D_u_cm_all.append(Du); D_w_cm_all.append(Dw)

    X_gm = np.concatenate([saved[f'gm_b{float(b):.2f}'] for b in gm_betas], axis=0)
    Du, Dw = distances_from_emb_stack(X_gm)
    D_u_gm_all.append(Du); D_w_gm_all.append(Dw)

means_emb = {
    'D_u_sbm': np.mean(D_u_sbm_all, axis=0), 'D_w_sbm': np.mean(D_w_sbm_all, axis=0),
    'D_u_cm':  np.mean(D_u_cm_all,  axis=0), 'D_w_cm':  np.mean(D_w_cm_all,  axis=0),
    'D_u_gm':  np.mean(D_u_gm_all,  axis=0), 'D_w_gm':  np.mean(D_w_gm_all,  axis=0),
}
print(f'Loaded {n_real_loaded} realizations.')


In [20]:
def _nmi_curve(D_u_list, D_w_list, params, n_g):
    """NMI curves over multiple realizations: compares param=0 vs each other param."""
    n_real = len(D_u_list)
    nmi_u = np.zeros((n_real, len(params)))
    nmi_w = np.zeros((n_real, len(params)))
    for r, (Du, Dw) in enumerate(zip(D_u_list, D_w_list)):
        for i in range(1, len(params)):
            idx   = np.concatenate([np.arange(0, n_g), np.arange(i * n_g, (i + 1) * n_g)])
            l_sub = np.concatenate([np.zeros(n_g), np.ones(n_g)])
            nmi_u[r, i] = NMI(ClusterNMF(Du[np.ix_(idx, idx)], k=2), l_sub)
            nmi_w[r, i] = NMI(ClusterNMF(Dw[np.ix_(idx, idx)], k=2), l_sub)
    return nmi_u, nmi_w

### NMI curves — averaged over 10 realizations (n=1,000)

`_nmi_curve` evaluates NMI for each binary classification task `(param_0 vs param_i)` across all realizations, using `ClusterNMF` with k=2. The shaded band shows ± 1 standard deviation across realizations.

In [ ]:
configs_nmi = [
    ('SBM', sbm_alphas, 'α', D_u_sbm_all, D_w_sbm_all),
    ('CM',  cm_alphas,  'α', D_u_cm_all,  D_w_cm_all),
    ('GM',  gm_betas,   'β', D_u_gm_all,  D_w_gm_all),
]

plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (name, params, xlabel, D_u_list, D_w_list) in zip(axes, configs_nmi):
    nmi_u, nmi_w = _nmi_curve(D_u_list, D_w_list, params, n_graphs)
    for arr, label, color, marker, linestyle in [
        (nmi_u, 'Unmatched',   '#278644', 'o', '-'),
        (nmi_w, 'Wasserstein', '#DD6722', 's', '--'),
    ]:
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        ax.plot(params, mean, label=label, marker=marker, color=color,
                linestyle=linestyle, linewidth=2.5, markersize=10)
        ax.fill_between(params,
                        np.clip(mean - std, 0, 1),
                        np.clip(mean + std, 0, 1),
                        alpha=0.15, color=color)

    ax.set_title(f'{name}: NMI vs {xlabel}', fontsize=23)
    ax.set_xlabel(xlabel, fontsize=21)
    ax.set_ylabel('NMI', fontsize=21)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_xticks(params)
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle='--', alpha=0.4, linewidth=1)
    ax.legend(frameon=False, loc='lower right', prop={'size': 15, 'weight': 'bold'})

plt.tight_layout()
plt.show()

In [ ]:
for name, params, xlabel, D_u_list, D_w_list in configs_nmi:
    fig, ax = plt.subplots(figsize=(8, 6))
    nmi_u, nmi_w = _nmi_curve(D_u_list, D_w_list, params, n_graphs)
    for arr, label, color, marker, linestyle in [
        (nmi_u, 'Unmatched',   '#278644', 'o', '-'),
        (nmi_w, 'Wasserstein', '#DD6722', 's', '--'),
    ]:
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        ax.plot(params, mean, label=label, marker=marker, color=color,
                linestyle=linestyle, linewidth=2, markersize=9)
        ax.fill_between(params,
                        np.clip(mean - std, 0, 1),
                        np.clip(mean + std, 0, 1),
                        alpha=0.15, color=color)
    ax.set_title(f'{name}: NMI vs {xlabel}', fontsize=20, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=16, fontweight='bold')
    ax.set_ylabel('NMI', fontsize=16, fontweight='bold')
    ax.set_xticks(params)
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.set_ylim(-0.05, 1.05)
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(frameon=False, loc='lower right', prop={'size': 17, 'weight': 'bold'})
    plt.tight_layout()
    plt.show()
 

---
## Section 5 — Large-graph Monte Carlo pipeline (n=10,000)

Scales up the experiment to **n=10,000** nodes. Because the full Gram matrix `X @ Xᵀ` is too large to sort exactly (10⁸ entries), the Wasserstein-2 distance is approximated via Monte Carlo: `n_mc = 500,000` directed node pairs `(i≠j)` are sampled per graph.

Embeddings and MC scalar products are cached to `output_mc2_emb/real_XX.npz`. The parameter grids `sbm_alphas2`, `cm_alphas2`, `gm_betas2` are refined subsets of the original grids, concentrated in the transition region between undetectable and detectable structure.

In [14]:
sbm_alphas2 = [0.94, 1.04, 1.16, 1.21, 1.28, 1.42, 1.58]
cm_alphas2  = [0.70, 1.06, 1.15, 1.31, 1.45, 1.61, 1.98]
gm_betas2   = [1.40, 1.64, 1.92, 2.10,2.25, 2.42, 2.63]

### MC helper functions

- **`mc_features_exact`** — samples exactly `n_mc` directed pairs `(i≠j)` and returns their sorted dot products alongside the normalised eigenvalue array.
- **`gm_large_mc`** — builds the Geometric Model adjacency in row-chunks of size `chunk` to avoid materialising the full n×n distance matrix.
- **`load_or_compute_mc`** — checkpointed driver: for each realization it computes only the `(model, param)` combinations absent from the NPZ, then writes back.

In [ ]:
n_nodes_mc = 10000  # graph size for the large-graph MC pipeline
n_mc       = 500000  # directed node pairs sampled per graph for MC Wasserstein

MAX_PARAMS_MC = 50


def mc_features_exact(X, n_mc_=None):
    """Eigenvalues + exactly n_mc sampled directed dot-products (i≠j), sorted."""
    if n_mc_ is None:
        n_mc_ = n_mc
    n = len(X)
    eigs = (np.linalg.eigvalsh(X.T @ X) / n).astype(np.float32)
    collected = []
    while len(collected) < n_mc_:
        need = (n_mc_ - len(collected)) * 2
        a = np.random.randint(0, n, need)
        b = np.random.randint(0, n, need)
        mask = a != b
        d = np.einsum('ij,ij->i', X[a[mask]], X[b[mask]])
        collected.extend(d.tolist())
    dots = np.sort(np.array(collected[:n_mc_], dtype=np.float32))
    return eigs, dots


def gm_large_mc(X_pos, c, beta, chunk=500):
    """Large-n Geometric Model: distances computed in row-chunks to avoid an n×n matrix."""
    n = len(X_pos)
    half_d = int(c / 2)
    no = np.einsum('ij,ij->i', X_pos, X_pos)
    fs, ss = [], []
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        D_c = np.sqrt(np.maximum(0.0,
                      no[start:end, None] + no[None, :] - 2 * X_pos[start:end] @ X_pos.T))
        P_c = np.exp(-beta * D_c)
        for k in range(end - start):
            i = start + k
            P_c[k, i] = 0.0
            p = P_c[k] / P_c[k].sum()
            nbrs = np.random.choice(n, half_d, p=p, replace=False)
            fs.extend([i] * half_d)
            ss.extend(nbrs.tolist())
    return pd.DataFrame({'i': fs + ss, 'j': ss + fs})


def _mc_for_sbm(a, seed):
    k2 = 2
    l2 = np.array([i // (n_nodes_mc // k2) for i in range(n_nodes_mc)], dtype=int)
    theta2 = np.ones(n_nodes_mc)
    c_in, c_out = compute_cin_cout(c, k2, a)
    C2 = np.ones((k2, k2)) * c_out + np.diag(np.ones(k2)) * (c_in - c_out)
    np.random.seed(seed)
    eigs_list, dots_list = [], []
    for _ in range(n_graphs):
        df  = DCSBM((C2, c, l2, theta2, symmetric, make_connected))
        emb = node_embedding(df_to_adj(df), dim); del df
        e, d = mc_features_exact(emb); del emb
        eigs_list.append(e); dots_list.append(d)
    return np.stack(eigs_list), np.stack(dots_list)


def _mc_for_cm(a, seed):
    np.random.seed(seed)
    eigs_list, dots_list = [], []
    for _ in range(n_graphs):
        df  = gen_cm_via_dcsbm((n_nodes_mc, c, a, symmetric, make_connected))
        emb = node_embedding(df_to_adj(df), dim); del df
        e, d = mc_features_exact(emb); del emb
        eigs_list.append(e); dots_list.append(d)
    return np.stack(eigs_list), np.stack(dots_list)


def _mc_for_gm(b, seed):
    np.random.seed(seed)
    r_xy = np.random.uniform(0, 1, n_nodes_mc)
    th   = np.random.uniform(0, 2 * np.pi, n_nodes_mc)
    X_pos = np.column_stack([r_xy * np.cos(th), r_xy * np.sin(th)])
    eigs_list, dots_list = [], []
    for _ in range(n_graphs):
        df  = gm_large_mc(X_pos, c, b)
        emb = node_embedding(df_to_adj(df), dim); del df
        e, d = mc_features_exact(emb); del emb
        eigs_list.append(e); dots_list.append(d)
    return np.stack(eigs_list), np.stack(dots_list)


def load_or_compute_mc(r, out_dir_mc):
    """Load MC features for realization r; compute and cache only missing (model, param) keys."""
    npz_path = os.path.join(out_dir_mc, f'real_{r:02d}.npz')
    saved = {}
    if os.path.exists(npz_path):
        saved = dict(np.load(npz_path, allow_pickle=True))

    updated = False

    for i, a in enumerate(sbm_alphas2):
        key_e, key_d = f'sbm_a{float(a):.4f}_eigs', f'sbm_a{float(a):.4f}_dots'
        if key_e not in saved:
            seed = r * 3 * MAX_PARAMS_MC + i
            print(f'  real_{r:02d}: SBM α={a}  (seed={seed})...')
            saved[key_e], saved[key_d] = _mc_for_sbm(a, seed)
            updated = True

    for i, a in enumerate(cm_alphas2):
        key_e, key_d = f'cm_a{float(a):.4f}_eigs', f'cm_a{float(a):.4f}_dots'
        if key_e not in saved:
            seed = r * 3 * MAX_PARAMS_MC + MAX_PARAMS_MC + i
            print(f'  real_{r:02d}: CM α={a}  (seed={seed})...')
            saved[key_e], saved[key_d] = _mc_for_cm(a, seed)
            updated = True

    for i, b in enumerate(gm_betas2):
        key_e, key_d = f'gm_b{float(b):.4f}_eigs', f'gm_b{float(b):.4f}_dots'
        if key_e not in saved:
            seed = r * 3 * MAX_PARAMS_MC + 2 * MAX_PARAMS_MC + i
            print(f'  real_{r:02d}: GM β={b}  (seed={seed})...')
            saved[key_e], saved[key_d] = _mc_for_gm(b, seed)
            updated = True

    if updated:
        np.savez_compressed(npz_path, **saved)
        print(f'  → saved {npz_path}')

    return saved

In [ ]:
n_real_mc  = 10
out_dir_mc = 'output_mc2_emb'
os.makedirs(out_dir_mc, exist_ok=True)

for r in tqdm(range(n_real_mc), desc='Realizations MC'):
    load_or_compute_mc(r, out_dir_mc)

print('Done.')

### Load MC distances

`distances_from_mc` reconstructs Unmatched and Wasserstein distance matrices directly from the cached eigenvalue and dot-product arrays, without re-embedding. Results across realizations are averaged into `means_mc`.

In [ ]:
def distances_from_mc(saved, model_prefix, params):
    """Unmatched and Wasserstein distance matrices from MC features stored in `saved`."""
    all_feats = []
    for a in params:
        key_e = f'{model_prefix}{float(a):.4f}_eigs'
        key_d = f'{model_prefix}{float(a):.4f}_dots'
        eigs_stack = saved[key_e]  # (n_graphs, dim)
        dots_stack = saved[key_d]  # (n_graphs, n_mc)
        for g in range(len(eigs_stack)):
            all_feats.append((eigs_stack[g], dots_stack[g]))

    N = len(all_feats)
    pairs = list(itertools.combinations(range(N), 2))

    eig_mat = np.stack([f[0] for f in all_feats])
    D_u2 = euclidean_distances(eig_mat)

    w2_vals2 = []
    for i, j in pairs:
        s1, s2 = all_feats[i][1], all_feats[j][1]
        w2_vals2.append(float(np.sqrt(np.mean((s1 - s2) ** 2))))

    return D_u2, squareform(w2_vals2)


files_mc = sorted(glob.glob(os.path.join(out_dir_mc, 'real_*.npz')))
n_mc_loaded = len(files_mc)

D_u_sbm_mc2, D_w_sbm_mc2 = [], []
D_u_cm_mc2,  D_w_cm_mc2  = [], []
D_u_gm_mc2,  D_w_gm_mc2   = [], []

for f in tqdm(files_mc, desc='Computing MC distances'):
    saved = dict(np.load(f, allow_pickle=True))
    Du, Dw = distances_from_mc(saved, 'sbm_a', sbm_alphas2)
    D_u_sbm_mc2.append(Du); D_w_sbm_mc2.append(Dw)
    Du, Dw = distances_from_mc(saved, 'cm_a', cm_alphas2)
    D_u_cm_mc2.append(Du); D_w_cm_mc2.append(Dw)
    Du, Dw = distances_from_mc(saved, 'gm_b', gm_betas2)
    D_u_gm_mc2.append(Du); D_w_gm_mc2.append(Dw)

means_mc = {
    'D_u_sbm': np.mean(D_u_sbm_mc2, axis=0), 'D_w_sbm': np.mean(D_w_sbm_mc2, axis=0),
    'D_u_cm':  np.mean(D_u_cm_mc2,  axis=0), 'D_w_cm':  np.mean(D_w_cm_mc2,  axis=0),
    'D_u_gm':  np.mean(D_u_gm_mc2,  axis=0), 'D_w_gm':  np.mean(D_w_gm_mc2,  axis=0),
}
print(f'Loaded {n_mc_loaded} realizations.')

### NMI curves — averaged over 10 realizations (n=10,000, MC Wasserstein)

Same `_nmi_curve` analysis as Section 4, applied to the large-graph MC pipeline. The combined panel plot and individual per-model figures follow.

In [ ]:
# Global font settings
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

configs_mc = [
    ('SBM', sbm_alphas2, 'α', D_u_sbm_mc2, D_w_sbm_mc2),
    ('CM',  cm_alphas2,  'α', D_u_cm_mc2,  D_w_cm_mc2),
    ('GM',  gm_betas2,   'β', D_u_gm_mc2,  D_w_gm_mc2),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (name, params, xlabel, D_u_list, D_w_list) in zip(axes, configs_mc):
    nmi_u, nmi_w = _nmi_curve(D_u_list, D_w_list, params, n_graphs)
    for arr, label, color, marker, linestyle in [
        (nmi_u, 'Unmatched',   '#278644', 'o', '-'),
        (nmi_w, 'Wasserstein', '#DD6722', 's', '--'),
    ]:
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        ax.plot(params, mean, label=label, marker=marker, color=color,
                linestyle=linestyle, linewidth=2.5, markersize=10)
        ax.fill_between(params,
                        np.clip(mean - std, 0, 1),
                        np.clip(mean + std, 0, 1),
                        alpha=0.15, color=color)
    ax.set_title(f'{name}: NMI vs {xlabel}', fontsize=23)
    ax.set_xlabel(xlabel, fontsize=21)
    ax.set_ylabel('NMI', fontsize=21)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_xticks(params)
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle='--', alpha=0.4, linewidth=1)
    ax.legend(frameon=False, loc='lower right', prop={'size': 15, 'weight': 'bold'})

plt.tight_layout()
plt.show()

In [ ]:
# Global font settings
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

configs_mc = [
    ('SBM', sbm_alphas2, 'α', D_u_sbm_mc2, D_w_sbm_mc2),
    ('CM',  cm_alphas2,  'α', D_u_cm_mc2,  D_w_cm_mc2),
    ('GM',  gm_betas2,   'β', D_u_gm_mc2,  D_w_gm_mc2),
]

for name, params, xlabel, D_u_list, D_w_list in configs_mc:
    fig, ax = plt.subplots(figsize=(8, 6))
    nmi_u2, nmi_w2 = _nmi_curve(D_u_list, D_w_list, params, n_graphs)
    for arr, label, color, marker, linestyle in [
        (nmi_u2, 'Unmatched',   '#278644', 'o', '-'),
        (nmi_w2, 'Wasserstein', '#DD6722', 's', '--'),
    ]:
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        ax.plot(params, mean, label=label, marker=marker, color=color,
                linestyle=linestyle, linewidth=2, markersize=9)
        ax.fill_between(params,
                        np.clip(mean - std, 0, 1),
                        np.clip(mean + std, 0, 1),
                        alpha=0.15, color=color)
    ax.set_title(f'{name}: NMI vs {xlabel}', fontsize=20, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=16, fontweight='bold')
    ax.set_ylabel('NMI', fontsize=16, fontweight='bold')
    ax.set_xticks(params)
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.set_ylim(-0.05, 1.05)
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(frameon=False, loc='lower right', prop={'size': 17, 'weight': 'bold'})
    plt.tight_layout()
    plt.show()
  